<a href="https://colab.research.google.com/github/claudia-miranda/pokemon-ecology/blob/main/Pokemon_Ecology_students_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Pokémon Safari Zone: From Continuum Modelling to Agent-Based Modelling

Welcome to this computational biology workshop! Today, we will build a spatial Agent-Based Model (ABM) to simulate how Pokémon move, grow, and compete in the Safari Zone.

But first of all... **What is an ABM?** An ABM is a computational model where a group of agents (in our case, pokemon) interact according to a set of rules. For example, they reproduce and die at a certain rate or they compete for food resources.

There are different ways of setting up ABMs. Today, we will learn how to do so starting from the ODEs that you all learnt about in the first part of the workshop.

### 🛠️ Part 1: Environment Setup
Run the code cell below to import the required Python packages (`numpy`, `scipy`, `matplotlib`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy.integrate import solve_ivp

# Set seed for reproducible workshop results
np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


### 🛠️ Part 2: Movement & Population Core Engine
In our ABM, the Pokémon can:
1. Move around
2. Reproduce
3. Die naturally
4. Die due to crowding or competition for resources like habitat or food.

To implement these rules, we need to build the following functions on Python:
1. **Movement (`update_positions`):** Pokémon move via 2D Brownian Motion with diffusion coefficient $D$ over time step $\Delta t = 0.1\text{ min}$.
2. **Birth & Death (`evolve_population`):** In each time step, Pokémon can faint (die) with probability $\delta_i \Delta t$ or reproduce with probability $b \Delta t$.
3. **Local Density (`get_local_densities`):** Vectorized calculation counting neighbours within sensing radius $\ell$.

Let's look at the Mathematics behind the implementation of these Python functions.

-----

#### **1. Movement: 2D Brownian Motion**
Pokémon move randomly through space via 2D Brownian motion (diffusion).

Mathematically, the position $\mathbf{x}_i(t) = (x_i(t), y_i(t))$ of Pokémon $i$ evolves over a small continuous time increment $dt$ according to the stochastic differential equation:
$$d\mathbf{x}_i = \sqrt{2D} \, d\mathbf{W}_i$$

where $D$ is the diffusion coefficient and $\mathbf{W}_i$ is a 2D Wiener process (standard Brownian motion). A stochastic differential equation is stochastic because it has a component that adds "randomness" to it. In this case, that is $\mathbf{W}_i$.

To implement this on a computer, we use the **Euler-Maruyama discretisation scheme** over a finite time step $\Delta t$:
$$x_i(t + \Delta t) = x_i(t) + \sqrt{2 D \Delta t} \cdot N(0, 1)$$
$$y_i(t + \Delta t) = y_i(t) + \sqrt{2 D \Delta t} \cdot N(0, 1)$$

where $N(0, 1)$ represents a random draw from a standard normal distribution (mean $0$, variance $1$).

In Python, $N(0, 1)$ is generated using `np.random.normal(size=num_pkm)`. The factor `step_scale = np.sqrt(2 * D * dt)` scales the random steps to accurately match the physical rate of diffusion.

----

#### **2. Birth & Death: Bernoulli Trial Probabilities**
In continuous time, an individual Pokémon reproduces at an intrinsic rate $b$ (events per unit time).

Over a small discrete time step $\Delta t$, the continuous rate translates into a probability $P_{\text{birth}}$:
$$P_{\text{birth}} = b \cdot \Delta t$$

##### Why do we choose $r_{\text{birth}} < b \cdot \Delta t$?
To decide whether a Pokémon reproduces during a single time step $\Delta t$, we simulate a **Bernoulli trial** (a random coin flip):
1. We draw a uniform random number $r_{\text{birth}} \sim U(0, 1)$, which yields a value between $0.0$ and $1.0$ with equal probability.
2. The probability that $r_{\text{birth}}$ falls inside the interval $[0, P_{\text{birth}}]$ is exactly equal to $P_{\text{birth}}$.
3. Therefore, checking if $r_{\text{birth}} < b \cdot \Delta t$ correctly triggers a birth event with the exact probability $b \cdot \Delta t$.

---

#### **3. Local Density Calculation**
`get_local_densities` counts the number of neighbours within sensing radius $\ell$ to adjust local mortality rates.

##### **Why use Local Mortality Rates ($\delta_i$)?**
In classical differential equations, we assume that individuals sense the whole area instantly (a "well-mixed" and homogeneous scheme: the mean-field approximation). However, real organisms only interact with nearby individuals due to limited sensing ranges, territorial competition, or resource sharing.

Local mortality rates allow us to capture spatial effects like crowding:
* A Pokémon isolated in an open field experiences a low baseline mortality rate $d_0$.
* A Pokémon trapped in a densely packed cluster experiences high local competition, drastically increasing its probability of fainting ($\delta_i$).

---

##### **How do we derive $\delta_i$ mathematically?**

1. **Calculate Local Density ($\rho_\ell^i$).**
   For a Pokémon $i$ located at $(x_i, y_i)$, we count the number of neighbours $N_\ell^i$ within a circular interaction disk of radius:
   $$\rho_\ell^i = \frac{N_\ell^i}{\pi \ell^2}$$

2. **Formulate the Individual Mortality Rate ($\delta_i$).**
   To translate local density $\rho_\ell^i$ into an effective mortality rate, we scale it by the total area $L^2 = L \times L$:
   $$\delta_i(t) = d_0 + d_1 \cdot \left( \rho_\ell^i \cdot L^2 \right)$$

   * **Single-Species (Pikachu Only):**
     $$\delta_{N}^i = d_0 + d_1 \left( \frac{N_\ell^i}{\pi \ell^2} \right) L^2$$
     where $d_1$ is the intra-species crowding penalty.

   * **Two-Species Competition (Pikachu vs. Charmander):**
     Now imagine there are also Charmanders around. When both species are present, Pikachu's mortality rate will depend on both its own species density ($\rho_{NN}^i$) and Charmanders density ($\rho_{NP}^i$):
     $$\delta_{N}^i = d_0 + \left( a_N \rho_{NN}^i + c_N \rho_{NP}^i \right) L^2$$
     where $a_N$ is self-competition and $c_N$ is cross-species penalty imposed by nearby Charmanders.

3. **Convert Rate to Individual Death Probability.**
   Just like the birth condition, individual $i$ faints if a uniform random number $r_{\text{death}} \sim {U}(0, 1)$ satisfies
   $$r_{\text{death}} < \delta_i(t) \cdot \Delta t$$.



👉 **Your Tasks:**
* **`TASK 1`**: Complete the updating step for $y$-coordinates using the Euler-Maruyama formula: `step_scale * np.random.normal(size=num_pkm)`.
* **`TASK 2`**: Complete the conditional check for birth: `r_birth < (b * dt)`.

In [ ]:
# --- GLOBAL SIMULATION PARAMETERS ---
L = 50.0          # Size of grid (50x50 meters)
dt = 0.1          # Time step in minutes (0.1 min = 6 seconds)
Tfinal = 60.0     # Total simulation time in minutes
steps = int(Tfinal / dt)
D = 1.0           # Diffusion speed (m^2 / min)

def update_positions(pos, D, dt, L):
    """Moves Pokémon using 2D Brownian Motion."""
    num_pkm = len(pos)
    if num_pkm == 0:
        return pos

    step_scale = np.sqrt(2 * D * dt)
    pos[:, 0] += step_scale * np.random.normal(size=num_pkm)

    # --------------------------------------------------------------------------
    # TASK 1: Complete the line for updating Y coordinates!
    # --------------------------------------------------------------------------
    # FIXME: Replace 'None' with the correct expression
    pos[:, 1] += None

    return pos % L

def evolve_population(pos, b, d_vector, dt, L, split_dist=0.2):
    """Evaluates birth and death events for individuals."""
    num_pkm = len(pos)
    if num_pkm == 0:
        return pos

    surviving_indices = []
    new_borns = []

    for i in range(num_pkm):
        r_birth = np.random.rand()
        r_death = np.random.rand()

        if r_death < (d_vector[i] * dt):
            continue  # Pokémon faints

        surviving_indices.append(i)

        # ----------------------------------------------------------------------
        # TASK 2: Fill in the proliferation condition! (birth if r_birth < b * dt)
        # ----------------------------------------------------------------------
        # FIXME: Replace 'False' with the correct condition

        if False:
            theta = np.random.rand() * 2 * np.pi
            child_x = pos[i, 0] + split_dist * np.cos(theta)
            child_y = pos[i, 1] + split_dist * np.sin(theta)
            new_borns.append([child_x, child_y])

    surviving_pos = pos[surviving_indices] if len(surviving_indices) > 0 else np.empty((0, 2))

    if len(new_borns) > 0:
        all_pos = np.vstack([surviving_pos, np.array(new_borns)])
    else:
        all_pos = surviving_pos

    return all_pos % L

def get_local_densities(pos1, pos2, l_radius, L):
    """Fast vectorized local density calculation."""
    n1, n2 = len(pos1), len(pos2)
    if n1 == 0 or n2 == 0:
        return np.zeros(n1)

    dx = np.abs(pos1[:, 0, None] - pos2[:, 0])
    dy = np.abs(pos1[:, 1, None] - pos2[:, 1])

    dx = np.minimum(dx, L - dx)
    dy = np.minimum(dy, L - dy)

    dist_matrix = np.sqrt(dx**2 + dy**2)
    counts = np.sum(dist_matrix < l_radius, axis=1)

    if np.array_equal(pos1, pos2):
        counts = np.maximum(0, counts - 1)

    return counts / (np.pi * l_radius**2)

print("Engine functions initialized!")

Run the cell below to check if your functions are working correctly!

In [ ]:
def test_student_functions():
    print("--------------------------------------------------")
    print("🔍 RUNNING UNIT TESTS ON STUDENT FUNCTIONS...")
    print("--------------------------------------------------")

    # --- Test 1: update_positions ---
    try:
        init_pos = np.array([[25.0, 25.0], [10.0, 10.0]])
        # Set fixed random seed for predictable Brownian step
        np.random.seed(0)
        moved_pos = update_positions(init_pos.copy(), D=1.0, dt=0.1, L=50.0)

        # Check if Y coordinates changed from initial positions
        if np.allclose(moved_pos[:, 1], init_pos[:, 1]):
            print("❌ TEST 1 FAILED: Y-coordinates did not update in update_positions(). Check TASK 1!")
        else:
            print("✅ TEST 1 PASSED: update_positions() successfully updates both X and Y positions.")
    except Exception as e:
        print(f"❌ TEST 1 ERROR in update_positions(): {e}")

    # --- Test 2: evolve_population ---
    try:
        # 10 Pokémon, high birth rate (b=100.0) so birth MUST occur, zero death
        test_pos = np.array([[20.0, 20.0] for _ in range(10)])
        d_vec = np.zeros(10)

        np.random.seed(0)
        new_pop = evolve_population(test_pos.copy(), b=100.0, d_vector=d_vec, dt=0.1, L=50.0)

        if len(new_pop) <= 10:
            print("❌ TEST 2 FAILED: No new Pokémon were born in evolve_population(). Check TASK 2!")
        else:
            print(f"✅ TEST 2 PASSED: evolve_population() generated new births! (Population grew from 10 to {len(new_pop)}).")
    except Exception as e:
        print(f"❌ TEST 2 ERROR in evolve_population(): {e}")

    # --- Test 3: get_local_densities ---
    try:
        # Place 3 Pokémon at identical positions (distance = 0)
        cluster = np.array([[10.0, 10.0], [10.0, 10.0], [10.0, 10.0]])
        test_l = 3.0

        # Each Pokémon should count 2 neighbours
        expected_density = 2.0 / (np.pi * test_l**2)
        densities = get_local_densities(cluster, cluster, l_radius=test_l, L=50.0)

        if densities is None or len(densities) == 0:
            print("❌ TEST 3 FAILED: get_local_densities() returned None or empty array. Check this function!")
        elif not np.allclose(densities, expected_density):
            print("❌ TEST 3 FAILED: get_local_densities() calculation incorrect. Check this function!")
        else:
            print("✅ TEST 3 PASSED: get_local_densities() accurately calculated local density!")
    except Exception as e:
        print(f"❌ TEST 3 ERROR in get_local_densities(): {e}")

    print("--------------------------------------------------")

# Execute the test suite
test_student_functions()

### 🟡 Part 3: Single Species Model (Pikachu Only)
We will first model Pikachu growing in isolation. Local crowding increases mortality according to:
$$\delta_{Pikachu}^i(t) = d_0 + d_1 \cdot \left(\rho_{NN}^i \cdot L^2\right)$$

In [ ]:
# Pikachu Single-Species Parameters
N0 = 10
b_N = 0.08      # Birth rate (per min)
d_0 = 0.01      # Baseline death rate (per min)
d_1 = 0.00005   # Intra-species crowding factor
l_N = 3.0       # Sensing radius (meters)

pikachu_pos = np.random.rand(N0, 2) * L

pika_history_pos = []
pika_history_counts = []

print("Running Part 1 Simulation (Pikachu Only)...")
for step in range(steps):
    n_pika = len(pikachu_pos)
    rho_NN = get_local_densities(pikachu_pos, pikachu_pos, l_N, L) if n_pika > 0 else np.zeros(0)
    d_pikachu = d_0 + d_1 * (rho_NN * L**2)

    pikachu_pos = update_positions(pikachu_pos, D, dt, L)
    pikachu_pos = evolve_population(pikachu_pos, b_N, d_pikachu, dt, L)

    pika_history_pos.append(pikachu_pos.copy())
    pika_history_counts.append(len(pikachu_pos))

print("Part 1 Simulation Complete!")

# Render Animation Video
sample_stride = 5
sampled_frames = range(0, steps, sample_stride)

fig1, (ax1_map, ax1_graph) = plt.subplots(1, 2, figsize=(12, 5))

def animate_part1(frame_idx):
    ax1_map.clear()
    ax1_graph.clear()

    pos = pika_history_pos[frame_idx]
    if len(pos) > 0:
        ax1_map.scatter(pos[:, 0], pos[:, 1], color='gold', edgecolor='black', s=50)
    ax1_map.set_xlim(0, L); ax1_map.set_ylim(0, L)
    ax1_map.set_title(f"Pikachu Safari Zone (t = {frame_idx * dt:.1f} min)")

    time_axis = np.arange(frame_idx + 1) * dt
    ax1_graph.plot(time_axis, pika_history_counts[:frame_idx + 1], color='gold', lw=2, label='Pikachu (ABM)')
    ax1_graph.set_xlim(0, Tfinal); ax1_graph.set_xlabel("Time (min)"); ax1_graph.set_ylabel("Population Count")
    ax1_graph.set_title("Single Species Logistic Growth")
    ax1_graph.grid(True, alpha=0.3)

anim1 = FuncAnimation(fig1, animate_part1, frames=sampled_frames, interval=50, repeat=False)
plt.close(fig1)
HTML(anim1.to_html5_video())

### 🔴 Part 4: Two-Species Competition (Pikachu vs Charmander)
Now we introduce Charmander!

👉 **Your Task:** Complete `TASK 3` in the code block below by calculating cross-species local densities:
* `rho_NP`: Density of Charmanders near each Pikachu.
* `rho_PN`: Density of Pikachus near each Charmander.

In [ ]:
# Two-Species Parameters
P0 = 10
b_P = 0.08
d_P = 0.01
l_P = 3.0

a_N, a_P = 0.0005, 0.0005  # Self-competition
c_N = 0.0030               # Penalty on Pikachu from Charmander
c_P = 0.0010               # Penalty on Charmander from Pikachu

pikachu_pos = np.random.rand(N0, 2) * L
charmander_pos = np.random.rand(P0, 2) * L

pika_history_pos2 = []
charm_history_pos2 = []
pika_counts2 = []
charm_counts2 = []

print("Running Part 2 Simulation (Co-Culture)...")
for step in range(steps):
    n_pika = len(pikachu_pos)
    n_char = len(charmander_pos)

    rho_NN = get_local_densities(pikachu_pos, pikachu_pos, l_N, L) if n_pika > 0 else np.zeros(0)
    rho_PP = get_local_densities(charmander_pos, charmander_pos, l_P, L) if n_char > 0 else np.zeros(0)

    # --------------------------------------------------------------------------
    # TASK 3: Compute cross-species local densities!
    # --------------------------------------------------------------------------
    # FIXME: Complete the lines for cross-densities
    rho_NP = None  # Charmander near Pikachu
    rho_PN = None  # Pikachu near Charmander

    d_pikachu = d_0 + (a_N * rho_NN + c_N * rho_NP) * (L**2)
    d_charmander = d_P + (a_P * rho_PP + c_P * rho_PN) * (L**2)

    pikachu_pos = update_positions(pikachu_pos, D, dt, L)
    charmander_pos = update_positions(charmander_pos, D, dt, L)

    pikachu_pos = evolve_population(pikachu_pos, b_N, d_pikachu, dt, L)
    charmander_pos = evolve_population(charmander_pos, b_P, d_charmander, dt, L)

    pika_history_pos2.append(pikachu_pos.copy())
    charm_history_pos2.append(charmander_pos.copy())
    pika_counts2.append(len(pikachu_pos))
    charm_counts2.append(len(charmander_pos))

print("Part 2 Simulation Complete!")

# Render Animation Video
fig2, (ax2_map, ax2_graph) = plt.subplots(1, 2, figsize=(12, 5))

def animate_part2(frame_idx):
    ax2_map.clear()
    ax2_graph.clear()

    pos_pika = pika_history_pos2[frame_idx]
    pos_charm = charm_history_pos2[frame_idx]

    if len(pos_pika) > 0:
        ax2_map.scatter(pos_pika[:, 0], pos_pika[:, 1], color='gold', edgecolor='black', s=50, label='Pikachu')
    if len(pos_charm) > 0:
        ax2_map.scatter(pos_charm[:, 0], pos_charm[:, 1], color='orangered', edgecolor='black', s=50, label='Charmander')

    ax2_map.set_xlim(0, L); ax2_map.set_ylim(0, L)
    ax2_map.set_title(f"Co-Culture Competition (t = {frame_idx * dt:.1f} min)")
    ax2_map.legend(loc="upper right")

    time_axis = np.arange(frame_idx + 1) * dt
    ax2_graph.plot(time_axis, pika_counts2[:frame_idx + 1], color='gold', lw=2, label='Pikachu')
    ax2_graph.plot(time_axis, charm_counts2[:frame_idx + 1], color='orangered', lw=2, label='Charmander')
    ax2_graph.set_xlim(0, Tfinal); ax2_graph.set_xlabel("Time (min)"); ax2_graph.set_ylabel("Population Count")
    ax2_graph.set_title("Two-Species Competitive Exclusion")
    ax2_graph.legend(loc="upper left")
    ax2_graph.grid(True, alpha=0.3)

anim2 = FuncAnimation(fig2, animate_part2, frames=sampled_frames, interval=50, repeat=False)
plt.close(fig2)
HTML(anim2.to_html5_video())

### 📐 Part 5: ODE Model vs ABM Comparison
Now let's test if the ODE matches our stochastic Agent-Based Model!

We solve the Lotka-Volterra competition system, i.e.,
$$\frac{dN}{dt} = b_N N - (d_N + a_N N + c_N P) N$$
$$\frac{dP}{dt} = b_P P - (d_P + a_P P + c_P N) P$$
using `scipy.integrate.solve_ivp`.

👉 **Your Task:** Complete `TASK 4` in the code block below by typing in the right hand side of the ODEs above.

In [ ]:
# Define ODE derivative function
def competition_ode(t, y, b_N, b_P, d_N, d_P, a_N, a_P, c_N, c_P):
    N, P = y
    # --------------------------------------------------------------------------
    # TASK 4: Input the ODEs expressions below
    # --------------------------------------------------------------------------
    # FIXME: Complete the RHS of dN/dt and dP/dt
    dN_dt = None
    dP_dt = None
    return [dN_dt, dP_dt]

In the code below, we run the ABM `num_runs` times. Taking the average of these runs reduces stochastic effects, which should help compare to the ODE numerical solution. However, there is also some other parameter(s) related to the geometry of the problem that determine if the ABM runs and the ODE model will agree or not.

👉 **Your Task:** Try and tweak the relevant geometric parameters and see if you can make the ABM and the ODE model agree. Hint: recall that, in the ODE model, we are assuming that Pokémon are able to sense instantaneously what is happening in the entire Safari area.

In [1]:
# ==============================================================================
# 50-RUN ENSEMBLE ABM vs. MEAN-FIELD ODE COMPARISON
# ==============================================================================

num_runs = 50

# --- GLOBAL SIMULATION PARAMETERS ---
L = 50.0          # Size of grid (50x50 meters)
dt = 0.1          # Time step in minutes (0.1 min = 6 seconds)
Tfinal = 60.0     # Total simulation time in minutes
steps = int(Tfinal / dt)
D = 1.0         # Diffusion speed (m^2 / min)

# Pikachu Parameters
N0 = 150
b_N = 0.08      # Birth rate (per min)
d_0 = 0.01      # Baseline death rate (per min)
d_1 = 0.00005   # Intra-species crowding factor
l_N = 3     # Sensing radius (meters)

# Charmander Parameters
P0 = 150
b_P = 0.08
d_P = 0.01
l_P = 3

# Interaction Parameters
a_N, a_P = 0.0005, 0.0005  # Self-competition
c_N = 0.0030               # Penalty on Pikachu from Charmander
c_P = 0.0010               # Penalty on Charmander from Pikachu

print(f"🔄 Running {num_runs} stochastic ABM simulations... Please wait a moment.")

# Arrays to store trajectory counts for all 50 runs
# Shape: (num_runs, steps)
all_pika_counts = np.zeros((num_runs, steps))
all_charm_counts = np.zeros((num_runs, steps))

# --- 1. RUN STOCHASTIC ENSEMBLE ---
for run in range(num_runs):
    # Re-initialize random positions for each simulation run
    pikachu_pos = np.random.rand(N0, 2) * L
    charmander_pos = np.random.rand(P0, 2) * L

    for step in range(steps):
        n_pika = len(pikachu_pos)
        n_char = len(charmander_pos)

        # Calculate local densities
        rho_NN = get_local_densities(pikachu_pos, pikachu_pos, l_N, L) if n_pika > 0 else np.zeros(0)
        rho_PP = get_local_densities(charmander_pos, charmander_pos, l_P, L) if n_char > 0 else np.zeros(0)
        rho_NP = get_local_densities(pikachu_pos, charmander_pos, l_N, L) if n_pika > 0 else np.zeros(0)
        rho_PN = get_local_densities(charmander_pos, pikachu_pos, l_P, L) if n_char > 0 else np.zeros(0)

        # Death rates
        d_pikachu = d_0 + (a_N * rho_NN + c_N * rho_NP) * (L**2)
        d_charmander = d_P + (a_P * rho_PP + c_P * rho_PN) * (L**2)

        # Move and evolve
        pikachu_pos = update_positions(pikachu_pos, D, dt, L)
        charmander_pos = update_positions(charmander_pos, D, dt, L)

        pikachu_pos = evolve_population(pikachu_pos, b_N, d_pikachu, dt, L)
        charmander_pos = evolve_population(charmander_pos, b_P, d_charmander, dt, L)

        # Save counts
        all_pika_counts[run, step] = len(pikachu_pos)
        all_charm_counts[run, step] = len(charmander_pos)

print("✅ 50 ABM Simulations Complete!")

# --- 2. CALCULATE MEAN AND STANDARD ERROR ---
mean_pika = np.mean(all_pika_counts, axis=0)
mean_charm = np.mean(all_charm_counts, axis=0)

sem_pika = np.std(all_pika_counts, axis=0) / np.sqrt(num_runs)
sem_charm = np.std(all_charm_counts, axis=0) / np.sqrt(num_runs)

# --- 3. SOLVE CONTINUOUS ODE MODEL COMPUTATIONALLY ---
t_span = (0, Tfinal)
t_eval = np.linspace(0, Tfinal, 200)
y0 = [N0, P0]

sol = solve_ivp(
    competition_ode, t_span, y0, t_eval=t_eval,
    args=(b_N, b_P, d_0, d_P, a_N, a_P, c_N, c_P)
)

# --- 4. PLOT COMPARISON ---
plt.figure(figsize=(11, 6))
time_axis = np.arange(steps) * dt

# Individual ABM Runs (Faint background lines)
for run in range(num_runs):
    plt.plot(time_axis, all_pika_counts[run], color='gold', alpha=0.08)
    plt.plot(time_axis, all_charm_counts[run], color='orangered', alpha=0.08)

# Ensemble Average Lines
plt.plot(time_axis, mean_pika, color='gold', lw=3, label=f'Pikachu Mean ({num_runs} ABM Runs)')
plt.fill_between(time_axis, mean_pika - 1.96 * sem_pika, mean_pika + 1.96 * sem_pika, color='gold', alpha=0.25)

plt.plot(time_axis, mean_charm, color='orangered', lw=3, label=f'Charmander Mean ({num_runs} ABM Runs)')
plt.fill_between(time_axis, mean_charm - 1.96 * sem_charm, mean_charm + 1.96 * sem_charm, color='orangered', alpha=0.25)

# Deterministic ODE Curves
plt.plot(sol.t, sol.y[0], color='darkgoldenrod', linestyle='--', lw=2.5, label='Pikachu (Deterministic ODE)')
plt.plot(sol.t, sol.y[1], color='maroon', linestyle='--', lw=2.5, label='Charmander (Deterministic ODE)')

plt.xlabel("Time (min)", fontsize=12)
plt.ylabel("Population Count", fontsize=12)
plt.title("Comparison: 50-Run Ensemble ABM Average vs. Mean-Field ODE", fontsize=14)
plt.legend(loc="upper left", fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

🔄 Running 50 stochastic ABM simulations... Please wait a moment.


NameError: name 'np' is not defined